In [1]:
import pandas as pd
#Loading the hindi dataset
file_path  = "C:\\Users\\Harsha\\Downloads\\stemmed_hindi.csv"
df_hindi = pd.read_csv(file_path)
print(len(df_hindi))

6400


In [2]:
df_hindi_01 = df_hindi.drop(df_hindi.columns[0], axis=1)

In [3]:
df_hindi = df_hindi_01
df_hindi

,Sentence,anger,disgust,fear,joy,neutral,sadness,surprise
0,कहना कि यह बारे पूरा तरह खुश,False,False,False,False,False,False,True
1,ठीक ठीक देखना जानना कि आज श्री अनुपयुक्त होना ...,False,False,False,True,False,False,False
2,ओह वह एक रहना,False,False,False,True,False,False,False
3,सब तुम कारण अद्भुत अद्भुत,False,False,False,True,False,False,False
4,जानना जानना भयभीत होना जाना भयभीत होना जाना वह...,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...
6395,साल का,False,False,False,False,True,False,False
6396,कौन होना,True,False,False,False,False,False,False
6397,तुम अपना जीवन जोखिम डालना एक सैंडविच लेना,False,False,False,True,False,False,False
6398,महान,False,False,False,True,False,False,False


In [4]:
emotion_columns_h = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise',]

In [5]:
df_hindi['Emotion'] = df_hindi[emotion_columns_h].idxmax(axis=1)

In [5]:
df_hindi['Emotion'] = df_hindi[emotion_columns_h].idxmax(axis=1)

In [6]:
df_hindi = df_hindi[['Sentence', 'Emotion']]

In [7]:
df_hindi

,Sentence,Emotion
0,कहना कि यह बारे पूरा तरह खुश,surprise
1,ठीक ठीक देखना जानना कि आज श्री अनुपयुक्त होना ...,joy
2,ओह वह एक रहना,joy
3,सब तुम कारण अद्भुत अद्भुत,joy
4,जानना जानना भयभीत होना जाना भयभीत होना जाना वह...,surprise
...,...,...
6395,साल का,neutral
6396,कौन होना,anger
6397,तुम अपना जीवन जोखिम डालना एक सैंडविच लेना,joy
6398,महान,joy


In [8]:
import pandas as pd



# Function to get the type of each element in the column
# Assuming df_english is a slice of another DataFrame

df_hindi.loc[:, 'type'] = df_hindi['Sentence'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_hindi['type'].value_counts()

print(type_counts)


type
str      6257
float     143
Name: count, dtype: int64


C:\Users\Harsha\AppData\Local\Temp\ipykernel_5400\2835744455.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hindi.loc[:, 'type'] = df_hindi['Sentence'].apply(lambda x: type(x).__name__)


In [9]:
float_data = df_hindi[df_hindi['Sentence'].apply(lambda x: isinstance(x, float))]
print(float_data)

     Sentence   Emotion   type
130       NaN   neutral  float
236       NaN  surprise  float
245       NaN   neutral  float
300       NaN   neutral  float
323       NaN   neutral  float
...       ...       ...    ...
6317      NaN   neutral  float
6324      NaN  surprise  float
6345      NaN       joy  float
6373      NaN   neutral  float
6385      NaN     anger  float

[143 rows x 3 columns]


In [10]:
df_hindi = df_hindi.dropna(subset=['Sentence'])

In [11]:
df_hindi

,Sentence,Emotion,type
0,कहना कि यह बारे पूरा तरह खुश,surprise,str
1,ठीक ठीक देखना जानना कि आज श्री अनुपयुक्त होना ...,joy,str
2,ओह वह एक रहना,joy,str
3,सब तुम कारण अद्भुत अद्भुत,joy,str
4,जानना जानना भयभीत होना जाना भयभीत होना जाना वह...,surprise,str
...,...,...,...
6395,साल का,neutral,str
6396,कौन होना,anger,str
6397,तुम अपना जीवन जोखिम डालना एक सैंडविच लेना,joy,str
6398,महान,joy,str


In [12]:
df_hindi.loc[:, 'type'] = df_hindi['Emotion'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_hindi['type'].value_counts()

print(type_counts)

type
str    6257
Name: count, dtype: int64


In [14]:
!pip install tiktoken

In [15]:
!pip install sentencepiece

In [16]:
!pip install transformers

In [16]:
!pip install transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import pandas as pd

# Load the Spanish-specific BERT tokenizer and model
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = False)
model =  AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)  # 7 emotion labels

# Assume your Spanish DataFrame is named df_spanish
# It should have columns 'Tweet' and 'Emotion'

# Convert the emotion labels to numerical values if not already done
label_mapping = {'anger': 0, 'disgust': 1, 'fear': 2, 
                 'joy': 3, 'neutral': 4, 
                 'sadness': 5, 'surprise': 6}

# Using .loc to avoid SettingWithCopyWarning
df_hindi.loc[:, 'Emotion'] = df_hindi['Emotion'].map(label_mapping)

if df_hindi['Emotion'].isnull().any():
    print("NaN values found in Emotion column after mapping. Please check the mapping.")

# Split data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_hindi['Sentence'].tolist(),
    df_hindi['Emotion'].tolist(),
    test_size=0.2,
    random_state=42
)

# Tokenize the data
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)

import numpy as np

# Ensure train_texts is a list of strings
print("Type of train_texts:", type(train_texts))
print("Number of training examples:", len(train_texts))
print("First 5 training examples:", train_texts[:5])

# Check for NaN values in train_texts
nan_texts = [text for text in train_texts if isinstance(text, float) and np.isnan(text)]
if nan_texts:
    print("NaN values found in training data:", len(nan_texts), "NaN values found.")

# Remove NaN values
train_texts = [text for text in train_texts if isinstance(text, str) and text.strip() != ""]

# Proceed to check for empty strings again
empty_texts = [text for text in train_texts if text.strip() == ""]
if empty_texts:
    print("Empty texts found in training data:", empty_texts)

# Tokenize the training and validation texts
train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)



# Custom Dataset class for PyTorch
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}  # Use clone().detach()
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # Ensure labels are of type long
        return item

# Create the Dataset objects
train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    eval_strategy="epoch",           
    save_strategy="epoch",           
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(eval_results)


Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Type of train_texts: <class 'list'>
Number of training examples: 5005
First 5 training examples: ['आप पास था', 'मैं मतलब तकनीकी रूप यहयह नियम खिलाफ या कुछ भी परहेजगगार', 'ओह निश्चित रूप बर्नोली सिद्धान्त न्यूटन का गति तीसरा नियम एक संयोजन', 'तुम कहना का टांग तोडना वाला बुरा आदमी', 'अच्छा तरह दोपहर लिए धन्यवाद']


Epoch,Training Loss,Validation Loss
1,No log,1.557836
2,1.649800,1.553154
3,1.649800,1.468160


{'eval_loss': 1.468159556388855, 'eval_runtime': 33.7369, 'eval_samples_per_second': 37.111, 'eval_steps_per_second': 2.342, 'epoch': 3.0}


In [17]:
# Evaluate the model on the validation set
eval_results = trainer.evaluate()

# Print the evaluation results
print("Evaluation results:", eval_results)

# Extract and print accuracy if available
if "eval_accuracy" in eval_results:
    print("Accuracy:", eval_results["eval_accuracy"])
else:
    print("Accuracy metric not found in evaluation results.")


Evaluation results: {'eval_loss': 1.468159556388855, 'eval_runtime': 31.6301, 'eval_samples_per_second': 39.583, 'eval_steps_per_second': 2.498, 'epoch': 3.0}
Accuracy metric not found in evaluation results.


In [19]:
from sklearn.metrics import accuracy_score

# Get predictions
predictions = trainer.predict(val_dataset)

# Convert logits to predicted labels (if necessary)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Calculate accuracy
accuracy = accuracy_score(val_labels, predicted_labels)  # Ensure val_labels are defined
print("Manual Accuracy:", accuracy)



Manual Accuracy: 0.4936102236421725


In [20]:
# Specify the directory where you want to save the model
model_save_path = "C:\\Users\\Harsha\\Downloads\\MMLL\\hindi"  # Change this to your desired path

# Save the trained model
trainer.save_model(model_save_path)

# Save the tokenizer
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to {model_save_path}")


Model and tokenizer saved to C:\Users\Harsha\Downloads\MMLL\hindi


In [1]:
import pandas as pd
#Loading the hindi dataset
file_path  = "C:\\Users\\Harsha\\Downloads\\stemmed_eng1.csv"
df_english = pd.read_csv(file_path)
print(len(df_english))

10983


In [2]:
#df_english = df_english.drop(df_english.columns[0], axis=1)

In [2]:
df_english

,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,worry payment problem may never joyce meyer,False,True,False,False,False,False,True,False,False,False,True
1,whatever decide make sure make,False,False,False,False,True,True,True,False,False,False,False
2,also help majority nfl coaching inept bill obr...,True,False,True,False,True,False,True,False,False,False,False
3,accept challenge literally even feel exhilarat...,False,False,False,False,True,False,True,False,False,False,False
4,roommate okay cant spell autocorrect,True,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
10978,do not make angry id laugh tweet,True,False,True,False,False,False,False,False,False,False,False
10979,excited watch tonight come NSw,False,False,False,False,True,False,True,False,False,False,False
10980,blah blah blah kyrie etc. leave boston real news,True,False,True,False,False,False,False,False,True,False,False
10981,wise never trust flock wolf,False,False,False,False,False,False,False,False,False,False,False


In [3]:
emotion_columns_e = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

In [4]:
df_english['Emotion'] = df_english[emotion_columns_e].idxmax(axis=1)

In [5]:
df_english = df_english[['Tweet', 'Emotion']]

In [6]:
df_english

,Tweet,Emotion
0,worry payment problem may never joyce meyer,anticipation
1,whatever decide make sure make,joy
2,also help majority nfl coaching inept bill obr...,anger
3,accept challenge literally even feel exhilarat...,joy
4,roommate okay cant spell autocorrect,anger
...,...,...
10978,do not make angry id laugh tweet,anger
10979,excited watch tonight come NSw,joy
10980,blah blah blah kyrie etc. leave boston real news,anger
10981,wise never trust flock wolf,anger


In [8]:
import pandas as pd



# Function to get the type of each element in the column
# Assuming df_english is a slice of another DataFrame

df_english.loc[:, 'type'] = df_english['Tweet'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_english['type'].value_counts()

print(type_counts)


type
str      10958
float       25
Name: count, dtype: int64


C:\Users\Harsha\AppData\Local\Temp\ipykernel_18456\3696596814.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_english.loc[:, 'type'] = df_english['Tweet'].apply(lambda x: type(x).__name__)


In [10]:
float_data = df_english[df_english['Tweet'].apply(lambda x: isinstance(x, float))]
print(float_data)

      Tweet       Emotion   type
418     NaN          fear  float
1930    NaN           joy  float
3085    NaN          fear  float
4134    NaN           joy  float
4351    NaN           joy  float
5980    NaN           joy  float
6420    NaN           joy  float
7078    NaN           joy  float
7093    NaN       sadness  float
7342    NaN           joy  float
7386    NaN      surprise  float
8259    NaN       sadness  float
8482    NaN         anger  float
8606    NaN         anger  float
8974    NaN         anger  float
9140    NaN           joy  float
9418    NaN         anger  float
9847    NaN           joy  float
9951    NaN  anticipation  float
10505   NaN           joy  float
10527   NaN           joy  float
10654   NaN       sadness  float
10867   NaN         anger  float
10908   NaN       disgust  float
10969   NaN         anger  float


In [11]:
df_english = df_english.dropna(subset=['Tweet'])

In [12]:
df_english

,Tweet,Emotion,type
0,worry payment problem may never joyce meyer,anticipation,str
1,whatever decide make sure make,joy,str
2,also help majority nfl coaching inept bill obr...,anger,str
3,accept challenge literally even feel exhilarat...,joy,str
4,roommate okay cant spell autocorrect,anger,str
...,...,...,...
10978,do not make angry id laugh tweet,anger,str
10979,excited watch tonight come NSw,joy,str
10980,blah blah blah kyrie etc. leave boston real news,anger,str
10981,wise never trust flock wolf,anger,str


In [13]:
import pandas as pd



# Function to get the type of each element in the column
# Assuming df_english is a slice of another DataFrame

df_english.loc[:, 'type'] = df_english['Emotion'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_english['type'].value_counts()

print(type_counts)


type
str    10958
Name: count, dtype: int64


In [14]:
!pip install transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import pandas as pd

# Load the Spanish-specific BERT tokenizer and model
model_name = "bert-base-uncased"  # You can also use "bert-base-cased" for case-sensitive processing
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=11) # 6 emotion labels

# Assume your Spanish DataFrame is named df_spanish
# It should have columns 'Tweet' and 'Emotion'

# Convert the emotion labels to numerical values if not already done
label_mapping = {'anger': 0, 'anticipation': 1, 'disgust': 2, 'fear': 3, 
                 'joy': 4, 'love': 5, 'optimism': 6, 'pessimism': 7, 
                 'sadness': 8, 'surprise': 9, 'trust': 10}

# Using .loc to avoid SettingWithCopyWarning
df_english.loc[:, 'Emotion'] = df_english['Emotion'].map(label_mapping)

if df_english['Emotion'].isnull().any():
    print("NaN values found in Emotion column after mapping. Please check the mapping.")

# Split data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_english['Tweet'].tolist(),
    df_english['Emotion'].tolist(),
    test_size=0.2,
    random_state=42
)

# Tokenize the data
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)

import numpy as np

# Ensure train_texts is a list of strings
print("Type of train_texts:", type(train_texts))
print("Number of training examples:", len(train_texts))
print("First 5 training examples:", train_texts[:5])

# Check for NaN values in train_texts
nan_texts = [text for text in train_texts if isinstance(text, float) and np.isnan(text)]
if nan_texts:
    print("NaN values found in training data:", len(nan_texts), "NaN values found.")

# Remove NaN values
train_texts = [text for text in train_texts if isinstance(text, str) and text.strip() != ""]

# Proceed to check for empty strings again
empty_texts = [text for text in train_texts if text.strip() == ""]
if empty_texts:
    print("Empty texts found in training data:", empty_texts)

# Tokenize the training and validation texts
train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)



# Custom Dataset class for PyTorch
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}  # Use clone().detach()
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # Ensure labels are of type long
        return item

# Create the Dataset objects
train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    eval_strategy="epoch",           
    save_strategy="epoch",           
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(eval_results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Type of train_texts: <class 'list'>
Number of training examples: 8766
First 5 training examples: ['offense mavu nnd favorite part dallas barbecue', 'soon possiblen', 'cant believe go work job voluntarily apply', 'peter radio piece ball pure class dig revenge total balance pro', 'lack make nn']


Epoch,Training Loss,Validation Loss
1,1.602700,1.271444
2,1.235100,1.246044
3,0.910200,1.378396


{'eval_loss': 1.3783960342407227, 'eval_runtime': 45.7636, 'eval_samples_per_second': 47.898, 'eval_steps_per_second': 2.994, 'epoch': 3.0}


In [15]:
from sklearn.metrics import accuracy_score

# Get predictions
predictions = trainer.predict(val_dataset)

# Convert logits to predicted labels (if necessary)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Calculate accuracy
accuracy = accuracy_score(val_labels, predicted_labels)  # Ensure val_labels are defined
print("Manual Accuracy:", accuracy)


Manual Accuracy: 0.5789233576642335


In [1]:
import pandas as pd
#Loading the hindi dataset
file_path  = "C:\\Users\\Harsha\\Downloads\\stemmed_spanish.csv"
df_spanish = pd.read_csv(file_path)
print(len(df_spanish))

5675


In [3]:
df_english

,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,detrs horrible desagradable ir ver él cerr ojo...,False,False,True,True,False,False,False,False,False,False,False
1,gustar idea solitario a ec arrasar louis gloria,True,False,False,False,True,False,False,False,False,False,False
2,comn menos comn,False,False,False,False,False,False,False,False,False,False,False
3,s pequeo minifundista mundo saber recibir fond...,True,False,True,False,False,False,False,False,False,False,False
4,empezar sonar alarma corte sirena tener llamar...,True,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
5670,hora respiracin empezar enloquecer,False,False,False,False,False,False,False,False,False,False,False
5671,ojo entender preocupacin realmente piba volver...,True,False,False,False,False,False,False,True,False,False,False
5672,miedo perder afortunado tener,False,False,False,True,False,False,True,False,True,False,False
5673,broma jams enojo l siempre coger dm,True,False,False,False,True,False,True,False,False,False,False


In [2]:
df_english = df_spanish.drop(df_spanish.columns[0], axis=1)

In [4]:
emotion_columns = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

In [5]:
df_english['Emotion'] = df_english[emotion_columns].idxmax(axis=1)

In [6]:
df_english = df_english[['Tweet', 'Emotion']]

In [7]:
df_english

,Tweet,Emotion
0,detrs horrible desagradable ir ver él cerr ojo...,disgust
1,gustar idea solitario a ec arrasar louis gloria,anger
2,comn menos comn,anger
3,s pequeo minifundista mundo saber recibir fond...,anger
4,empezar sonar alarma corte sirena tener llamar...,anger
...,...,...
5670,hora respiracin empezar enloquecer,anger
5671,ojo entender preocupacin realmente piba volver...,anger
5672,miedo perder afortunado tener,fear
5673,broma jams enojo l siempre coger dm,anger


In [8]:
!pip install transformers scikit-learn pandas torch

In [9]:
!pip install transformers torch datasets

In [10]:
df_english


,Tweet,Emotion
0,detrs horrible desagradable ir ver él cerr ojo...,disgust
1,gustar idea solitario a ec arrasar louis gloria,anger
2,comn menos comn,anger
3,s pequeo minifundista mundo saber recibir fond...,anger
4,empezar sonar alarma corte sirena tener llamar...,anger
...,...,...
5670,hora respiracin empezar enloquecer,anger
5671,ojo entender preocupacin realmente piba volver...,anger
5672,miedo perder afortunado tener,fear
5673,broma jams enojo l siempre coger dm,anger


In [11]:
df_spanish = df_english

In [12]:
df_spanish

,Tweet,Emotion
0,detrs horrible desagradable ir ver él cerr ojo...,disgust
1,gustar idea solitario a ec arrasar louis gloria,anger
2,comn menos comn,anger
3,s pequeo minifundista mundo saber recibir fond...,anger
4,empezar sonar alarma corte sirena tener llamar...,anger
...,...,...
5670,hora respiracin empezar enloquecer,anger
5671,ojo entender preocupacin realmente piba volver...,anger
5672,miedo perder afortunado tener,fear
5673,broma jams enojo l siempre coger dm,anger


In [13]:
import pandas as pd



# Function to get the type of each element in the column
# Assuming df_english is a slice of another DataFrame

df_spanish.loc[:, 'type'] = df_spanish['Tweet'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_spanish['type'].value_counts()

print(type_counts)


type
str      5665
float      10
Name: count, dtype: int64


C:\Users\Harsha\AppData\Local\Temp\ipykernel_6224\168669242.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spanish.loc[:, 'type'] = df_spanish['Tweet'].apply(lambda x: type(x).__name__)


In [14]:
float_data = df_spanish[df_spanish['Tweet'].apply(lambda x: isinstance(x, float))]
print(float_data)

     Tweet   Emotion   type
229    NaN  surprise  float
869    NaN     anger  float
1034   NaN       joy  float
1073   NaN     anger  float
2412   NaN       joy  float
3399   NaN   sadness  float
3908   NaN     anger  float
4827   NaN     anger  float
4932   NaN     anger  float
5223   NaN     anger  float


In [15]:
df_spanish = df_spanish.dropna(subset=['Tweet'])

In [16]:
df_spanish

,Tweet,Emotion,type
0,detrs horrible desagradable ir ver él cerr ojo...,disgust,str
1,gustar idea solitario a ec arrasar louis gloria,anger,str
2,comn menos comn,anger,str
3,s pequeo minifundista mundo saber recibir fond...,anger,str
4,empezar sonar alarma corte sirena tener llamar...,anger,str
...,...,...,...
5670,hora respiracin empezar enloquecer,anger,str
5671,ojo entender preocupacin realmente piba volver...,anger,str
5672,miedo perder afortunado tener,fear,str
5673,broma jams enojo l siempre coger dm,anger,str


In [17]:
df_spanish.loc[:, 'type'] = df_spanish['Emotion'].apply(lambda x: type(x).__name__)


# Get the count of each type
type_counts = df_spanish['type'].value_counts()

print(type_counts)

type
str    5665
Name: count, dtype: int64


In [18]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import pandas as pd

# Load the Spanish-specific BERT tokenizer and model
model_name = 'dccuchile/bert-base-spanish-wwm-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=11)  # 11 emotion labels

# Assume your Spanish DataFrame is named df_spanish
# It should have columns 'Tweet' and 'Emotion'

# Convert the emotion labels to numerical values if not already done
label_mapping = {'anger': 0, 'anticipation': 1, 'disgust': 2, 'fear': 3, 
                 'joy': 4, 'love': 5, 'optimism': 6, 'pessimism': 7, 
                 'sadness': 8, 'surprise': 9, 'trust': 10}

# Using .loc to avoid SettingWithCopyWarning
df_spanish.loc[:, 'Emotion'] = df_spanish['Emotion'].map(label_mapping)

if df_spanish['Emotion'].isnull().any():
    print("NaN values found in Emotion column after mapping. Please check the mapping.")

# Split data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_spanish['Tweet'].tolist(),
    df_spanish['Emotion'].tolist(),
    test_size=0.2,
    random_state=42
)

# Tokenize the data
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)

import numpy as np

# Ensure train_texts is a list of strings
print("Type of train_texts:", type(train_texts))
print("Number of training examples:", len(train_texts))
print("First 5 training examples:", train_texts[:5])

# Check for NaN values in train_texts
nan_texts = [text for text in train_texts if isinstance(text, float) and np.isnan(text)]
if nan_texts:
    print("NaN values found in training data:", len(nan_texts), "NaN values found.")

# Remove NaN values
train_texts = [text for text in train_texts if isinstance(text, str) and text.strip() != ""]

# Proceed to check for empty strings again
empty_texts = [text for text in train_texts if text.strip() == ""]
if empty_texts:
    print("Empty texts found in training data:", empty_texts)

# Tokenize the training and validation texts
train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)



# Custom Dataset class for PyTorch
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}  # Use clone().detach()
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # Ensure labels are of type long
        return item

# Create the Dataset objects
train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    eval_strategy="epoch",           
    save_strategy="epoch",           
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(eval_results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Type of train_texts: <class 'list'>
Number of training examples: 4532
First 5 training examples: ['envidia sano estar serio', 'hoy atender mozo q gust llegu casa cmo mierda hacer averige nombre apellido gusto musical', 's si situacin sismo cosa si necesitar alguien escuchar aqu mucho fuerza', 'horror q diente tanto amarillo q cara tanto feo adems ms bello q tú', 'revista exponer mircoles congreso organizado uba']


Epoch,Training Loss,Validation Loss
1,No log,1.485162
2,1.601200,1.393130
3,1.601200,1.360799


{'eval_loss': 1.3607985973358154, 'eval_runtime': 36.0107, 'eval_samples_per_second': 31.463, 'eval_steps_per_second': 1.972, 'epoch': 3.0}


In [19]:
# Evaluate the model on the validation set
eval_results = trainer.evaluate()

# Print the evaluation results
print("Evaluation results:", eval_results)

# Extract and print accuracy if available
if "eval_accuracy" in eval_results:
    print("Accuracy:", eval_results["eval_accuracy"])
else:
    print("Accuracy metric not found in evaluation results.")


Evaluation results: {'eval_loss': 1.3607985973358154, 'eval_runtime': 36.2084, 'eval_samples_per_second': 31.291, 'eval_steps_per_second': 1.961, 'epoch': 3.0}
Accuracy metric not found in evaluation results.


In [20]:
from sklearn.metrics import accuracy_score

# Get predictions
predictions = trainer.predict(val_dataset)

# Convert logits to predicted labels (if necessary)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Calculate accuracy
accuracy = accuracy_score(val_labels, predicted_labels)  # Ensure val_labels are defined
print("Manual Accuracy:", accuracy)


Manual Accuracy: 0.589585172109444


In [22]:
# Specify the directory where you want to save the model
model_save_path = "C:\\Users\\Harsha\\Downloads\\MMLL"  # Change this to your desired path

# Save the trained model
trainer.save_model(model_save_path)

# Save the tokenizer
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to {model_save_path}")



Model and tokenizer saved to C:\Users\Harsha\Downloads\MMLL


In [18]:
!pip install transformers[torch]

  Using cached accelerate-1.0.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.0.1-py3-none-any.whl (330 kB)
